In [ ]:
# 1. 环境准备
!rm -rf /kaggle/working/Real-ESRGAN
!git clone --depth 1 --branch master https://github.com/aksjfds/Real-ESRGAN.git /kaggle/working/Real-ESRGAN
!pip install -q -r /kaggle/working/Real-ESRGAN/requirements.txt
!cd /kaggle/working/Real-ESRGAN && python -m py_compile inference.py inference/*.py encode/*.py audio/*.py inference/models/*.py
!cd /kaggle/working/Real-ESRGAN && python inference.py --help >/dev/null
!cd /kaggle/working/Real-ESRGAN && python -m audio.process --help >/dev/null
!ffmpeg -hide_banner -encoders 2>/dev/null | grep -E "(av1_nvenc|hevc_nvenc)" || true


In [ ]:
# 2. v8.7 [Dev] 配置

# ===== 输入 / 输出 / 测试范围 =====
INPUT_VIDEO = "/kaggle/input/datasets/rustacean1/hanime/ts_2.mp4"
OUTPUT_VIDEO = "/kaggle/working/realesrgan.mp4"
START_TIME = 5 * 60 + 35
TEST_SECONDS = 15

# ===== 视频增强 =====
VIDEO_ENHANCE = True
MODEL = "realesr-animevideov3"
MODEL_PATH = ""
SCALE = 2
DEBAND_STRENGTH = 0.006  # 0=关闭；FFmpeg deband 阈值，有效范围 0.00003-0.5
RIFE_FPS = 60  # 0=关闭 RIFE；>0 时必须 >= 源帧率

# GPU：False=cuda:0；True=cuda:0,1
DUAL_GPU = True

# BasicVSR++ 固定参数
BVS_TILE_SIZE = 640
BVS_CLIP_LENGTH = 13
BVS_BATCH_SIZE = 1
BVS_STRENGTH = 1.0

# ===== RTX 4090 NVENC：AV1 / HEVC =====
VIDEO_CODEC = "av1_nvenc"  # av1_nvenc / hevc_nvenc
ENCODE_GPU = 0
PRESET = "P7"
CQ = 18

# ===== AV1 NVENC 专属参数（VIDEO_CODEC="av1_nvenc" 时使用） =====
TUNE = "HQ"
PROFILE = "MAIN"
AV1_BIT_DEPTH = 8  # 8=8-bit片源保持8-bit；10=8-bit片源升为10-bit；10-bit片源始终输出10-bit
RC = "VBR"
BITRATE = "0"
MULTIPASS = "FULLRES"
B_FRAMES = 3
RC_LOOKAHEAD = 28     # SDK 13.1：最大 31 - B_FRAMES
SPATIAL_AQ = 1
TEMPORAL_AQ = 1
AQ_STRENGTH = 8
B_REF_MODE = "MIDDLE"
GOP_SIZE = 240

# HEVC NVENC 使用现有 backend 的 HQ / VBR / fullres multipass / AQ / lookahead 配置；
# 位深跟随推理帧：8-bit -> yuv420p，10-bit -> p010le。

# ===== 音频 =====
AUDIO_ENHANCE = True
AUDIO_CODEC = "aac"   # 增强开启时使用；关闭时自动 stream copy
AUDIO_BITRATE = "256k"


In [ ]:
# 3. 执行
import subprocess
import sys

def add_options(command, options):
    for key, value in options.items():
        command.extend([key, str(value)])

effective_audio_codec = AUDIO_CODEC if AUDIO_ENHANCE else "copy"

if VIDEO_CODEC not in {"av1_nvenc", "hevc_nvenc"}:
    raise ValueError(f"Unsupported VIDEO_CODEC in Notebook: {VIDEO_CODEC}")

common_options = {
    "--input": INPUT_VIDEO,
    "--output": OUTPUT_VIDEO,
    "--audio-codec": effective_audio_codec,
    "--audio-bitrate": AUDIO_BITRATE,
    "--start-time": START_TIME,
    "--test-seconds": TEST_SECONDS,
    "--ffmpeg-bin": "ffmpeg",
    "--ffprobe-bin": "ffprobe",
}

if VIDEO_ENHANCE:
    command = [sys.executable, "/kaggle/working/Real-ESRGAN/inference.py"]
    add_options(command, common_options)
    video_options = {
        "--model": MODEL,
        "--model-path": MODEL_PATH,
        "--scale": SCALE,
        "--deband-strength": DEBAND_STRENGTH,
        "--rife-fps": RIFE_FPS,
        "--gpu-ids": "0,1" if DUAL_GPU else "0",
        "--bvs-tile-size": BVS_TILE_SIZE,
        "--bvs-clip-length": BVS_CLIP_LENGTH,
        "--bvs-batch-size": BVS_BATCH_SIZE,
        "--bvs-strength": BVS_STRENGTH,
        "--video-codec": VIDEO_CODEC,
        "--cq": CQ,
        "--nvenc-preset": PRESET.lower(),
        "--encode-gpu": ENCODE_GPU,
    }
    if VIDEO_CODEC == "av1_nvenc":
        video_options.update({
            "--av1-profile": PROFILE.lower(),
            "--av1-bit-depth": AV1_BIT_DEPTH,
            "--av1-tune": TUNE.lower(),
            "--av1-rc": RC.lower(),
            "--av1-bitrate": BITRATE,
            "--av1-multipass": MULTIPASS.lower(),
            "--av1-rc-lookahead": RC_LOOKAHEAD,
            "--av1-spatial-aq": SPATIAL_AQ,
            "--av1-temporal-aq": TEMPORAL_AQ,
            "--av1-aq-strength": AQ_STRENGTH,
            "--av1-b-ref-mode": B_REF_MODE.lower(),
            "--av1-b-frames": B_FRAMES,
            "--av1-gop-size": GOP_SIZE,
        })
    add_options(command, video_options)
    cwd = None
else:
    command = [sys.executable, "-m", "audio.process"]
    add_options(command, common_options)
    cwd = "/kaggle/working/Real-ESRGAN"

if AUDIO_ENHANCE:
    command.append("--audio-enhance")

process = subprocess.Popen(
    command,
    cwd=cwd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    sys.stdout.write(line)
    sys.stdout.flush()

returncode = process.wait()
if returncode != 0:
    raise subprocess.CalledProcessError(returncode, command)
